In [ ]:
# XGBoost, CatBoost, Soft Voting ensemble on the two, penalised LR
# --- Step 0: Imports ---
import numpy as np
import pandas as pd
import re

from sklearn.model_selection import (
    train_test_split,
    StratifiedKFold,
    RandomizedSearchCV
)
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import (
    roc_auc_score,
    accuracy_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay
)
from sklearn.linear_model import LogisticRegressionCV
from sklearn.ensemble import VotingClassifier

import xgboost as xgb
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier

import matplotlib.pyplot as plt

In [ ]:
# --- Step 1: Load Data (or placeholder) ---
file_path = '/kaggle/input/predict-ovarian-cancer/Supplementary data 1.xlsx'
try:
    df = pd.read_excel(file_path)
    print(f"Loaded data from {file_path}")
    if 'SUBJECT_ID' in df.columns:
        df.drop('SUBJECT_ID', axis=1, inplace=True)
except FileNotFoundError:
    print(f"File not found at {file_path}, generating placeholder data.")
    rng = np.random.default_rng(42)
    n_samples = 500
    hematological_features = [
        'BASO#','BASO%','EO#','EO%','HCT','HGB','LYM#','LYM%','MCH','MCV',
        'MONO#','MONO%','MPV','NEU','PCT','PDW','PLT','RBC','RDW'
    ]
    clinical_features = [
        'AG','Age','ALB','ALP','ALT','AST','BUN','Ca','CL','CO2CP','CREA',
        'DBIL','GGT','GLO','GLU.','IBIL','K','Menopause','Mg','Na','PHOS',
        'TBIL','TP','UA'
    ]
    all_feats = hematological_features + clinical_features
    data = {c: rng.random(n_samples) * 100 for c in all_feats}
    for c in all_feats:
        if rng.random() < 0.1:
            idx = rng.choice(n_samples, size=int(0.05 * n_samples), replace=False)
            data[c][idx] = np.nan
    data['TYPE'] = rng.integers(0, 2, n_samples)
    df = pd.DataFrame(data)

# --- Step 2: Ensure feature columns exist ---
hematological_features = [
    'BASO#','BASO%','EO#','EO%','HCT','HGB','LYM#','LYM%','MCH','MCV',
    'MONO#','MONO%','MPV','NEU','PCT','PDW','PLT','RBC','RDW'
]
clinical_features = [
    'AG','Age','ALB','ALP','ALT','AST','BUN','Ca','CL','CO2CP','CREA',
    'DBIL','GGT','GLO','GLU.','IBIL','K','Menopause','Mg','Na','PHOS',
    'TBIL','TP','UA'
]
clin_hem_features = hematological_features + clinical_features

for feat in clin_hem_features:
    if feat not in df.columns:
        df[feat] = np.nan
if 'TYPE' not in df.columns:
    df['TYPE'] = np.random.randint(0, 2, size=len(df))

# --- Step 3: Clean object‐typed numeric columns ---
def clean_numeric_columns(df_in, columns):
    df_out = df_in.copy()
    def clean_value(x):
        if pd.isna(x): return np.nan
        if isinstance(x, (int, float)): return float(x)
        if isinstance(x, str):
            s = re.sub(r'\t|\s+', '', x)
            try: return float(s)
            except: return np.nan
        return np.nan
    for col in columns:
        if col in df_out.columns:
            df_out[col] = df_out[col].apply(clean_value).astype(float)
    return df_out

obj_cols = df[clin_hem_features].select_dtypes(include=['object']).columns.tolist()
if obj_cols:
    df = clean_numeric_columns(df, obj_cols)
    print("Cleaned object columns:", obj_cols)

# --- Step 4: Impute missing values ---
BIOMARKER_DISTS = {
    'AG':{'mean':12,'std':2},
    'ALB':{'mean':4.4,'std':0.5},
    'ALP':{'mean':80,'std':30},
    'ALT':{'mean':30,'std':12},
    'AST':{'mean':28,'std':10},
    'CO2CP':{'mean':26,'std':1.5},
    'DBIL':{'mean':0.15,'std':0.1},
    'GGT':{'mean':15,'std':7},
    'GLO':{'mean':2.9,'std':0.3},
    'IBIL':{'mean':0.5,'std':0.2},
    'MPV':{'mean':9.5,'std':1},
    'NEU':{'mean':4.75,'std':1.5},
    'PCT':{'mean':0.1,'std':0.2},
    'PDW':{'mean':13,'std':2},
    'TBIL':{'mean':0.75,'std':0.3},
    'TP':{'mean':7.15,'std':0.5}
}

def impute_normal(df_in, dist_dict, seed=42):
    df_out = df_in.copy()
    np.random.seed(seed)
    for marker, params in dist_dict.items():
        if marker in df_out.columns and df_out[marker].isna().any():
            mask = df_out[marker].isna()
            n_missing = mask.sum()
            vals = np.random.normal(params['mean'], params['std'], size=n_missing)
            vals[vals < 0] = 0
            df_out.loc[mask, marker] = vals
    return df_out

df = impute_normal(df, BIOMARKER_DISTS)
for col in clin_hem_features:
    if df[col].isna().any():
        m = df[col].mean()
        df[col].fillna(m if not np.isnan(m) else 0, inplace=True)

# --- Step 5: Prepare data, split, & scale ---
features = [f for f in clin_hem_features if f in df.columns]
X = df[features]
y = df['TYPE']

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

scaler = MinMaxScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

# Compute scale_pos_weight for imbalance
scale_pos_weight = (len(y_train) - y_train.sum()) / y_train.sum()
print("scale_pos_weight =", scale_pos_weight)

# CV splitter
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)


In [ ]:
# --- Step 6: XGBoost hyperparameter search & evaluation ---
xgb_param_dist = {
    'n_estimators':      [100, 300, 500, 800],
    'learning_rate':     [0.01, 0.05, 0.1, 0.2],
    'max_depth':         [3, 4, 5, 6],
    'subsample':         [0.6, 0.8, 1.0],
    'colsample_bytree':  [0.6, 0.8, 1.0],
    'gamma':             [0, 0.1, 0.2, 0.5],
    'reg_alpha':         [0, 0.01, 0.1, 1.0],
    'reg_lambda':        [1, 1.5, 2.0, 3.0],
    'scale_pos_weight':  [scale_pos_weight, 1]
}

xgb_base = XGBClassifier(
    objective='binary:logistic',
    use_label_encoder=False,
    eval_metric='auc',
    random_state=42
)

xgb_search = RandomizedSearchCV(
    xgb_base,
    param_distributions=xgb_param_dist,
    n_iter=40,
    scoring='balanced_accuracy',
    cv=cv,
    n_jobs=-1,
    verbose=2,
    random_state=42
)

print("Searching XGBoost hyperparameters...")
xgb_search.fit(X_train_scaled, y_train)
best_xgb = xgb_search.best_estimator_
print(f"Best XGB CV balanced_accuracy: {xgb_search.best_score_:.4f}")
print("Best XGB params:", xgb_search.best_params_)

# Evaluate
y_proba_xgb = best_xgb.predict_proba(X_test_scaled)[:,1]
# threshold tuning
thresholds = np.linspace(0.1, 0.9, 17)
best_thr, best_bal = 0.5, 0
for thr in thresholds:
    preds = (y_proba_xgb > thr).astype(int)
    bal = balanced_accuracy_score(y_test, preds)
    if bal > best_bal:
        best_bal, best_thr = bal, thr

y_pred_xgb = (y_proba_xgb > best_thr).astype(int)
print("\nXGBoost Metrics:")
print("AUC:              ", roc_auc_score(y_test, y_proba_xgb))
print("Balanced Accuracy:", balanced_accuracy_score(y_test, y_pred_xgb))
print(classification_report(y_test, y_pred_xgb, target_names=['Benign','Ovarian'], zero_division=0))

# Plot XGBoost confusion matrix
cm = confusion_matrix(y_test, y_pred_xgb)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['Benign','Ovarian'])
disp.plot(cmap=plt.cm.Blues)
plt.title('XGBoost Confusion Matrix')
plt.show()


In [ ]:
# --- Step 8: CatBoost hyperparameter search & evaluation ---
cat_param_dist = {
    'iterations':       [200, 500, 800],
    'learning_rate':    [0.01, 0.05, 0.1],
    'depth':            [4, 6, 8],
    'l2_leaf_reg':      [1, 3, 5, 7],
    'border_count':     [32, 64, 128],
    'scale_pos_weight': [scale_pos_weight, 1]
}

cat_base = CatBoostClassifier(
    loss_function='Logloss',
    eval_metric='AUC',
    random_seed=42,
    verbose=0
)

cat_search = RandomizedSearchCV(
    cat_base,
    param_distributions=cat_param_dist,
    n_iter=30,
    scoring='balanced_accuracy',
    cv=cv,
    n_jobs=-1,
    verbose=2,
    random_state=42
)

print("\nSearching CatBoost hyperparameters...")
cat_search.fit(X_train_scaled, y_train)
cat_best = cat_search.best_estimator_
print(f"Best CatBoost CV balanced_accuracy: {cat_search.best_score_:.4f}")
print("Best CatBoost params:", cat_search.best_params_)

# Evaluate
y_proba_cat = cat_best.predict_proba(X_test_scaled)[:,1]
y_pred_cat  = (y_proba_cat > 0.5).astype(int)
print("\nCatBoost Metrics:")
print("AUC:              ", roc_auc_score(y_test, y_proba_cat))
print("Balanced Accuracy:", balanced_accuracy_score(y_test, y_pred_cat))
print(classification_report(y_test, y_pred_cat, target_names=['Benign','Ovarian'], zero_division=0))

# Plot CatBoost confusion matrix
cm = confusion_matrix(y_test, y_pred_cat)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['Benign','Ovarian'])
disp.plot(cmap=plt.cm.Blues)
plt.title('CatBoost Confusion Matrix')
plt.show()


In [ ]:
# --- Step 9: Penalized Logistic Regression Baseline & confusion matrix ---
logreg = LogisticRegressionCV(
    Cs=10,
    cv=cv,
    penalty='elasticnet',
    solver='saga',
    l1_ratios=[0.5],
    scoring='balanced_accuracy',
    class_weight='balanced',
    max_iter=10000,
    random_state=42,
    n_jobs=-1,
    refit=True
)

print("\nTraining LogisticRegressionCV...")
logreg.fit(X_train_scaled, y_train)
y_proba_lr = logreg.predict_proba(X_test_scaled)[:,1]
y_pred_lr  = (y_proba_lr > 0.5).astype(int)

print("\nLogistic Regression Metrics:")
print("AUC:              ", roc_auc_score(y_test, y_proba_lr))
print("Balanced Accuracy:", balanced_accuracy_score(y_test, y_pred_lr))
print(classification_report(y_test, y_pred_lr, target_names=['Benign','Ovarian'], zero_division=0))

# Plot Logistic Regression confusion matrix
cm = confusion_matrix(y_test, y_pred_lr)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['Benign','Ovarian'])
disp.plot(cmap=plt.cm.Blues)
plt.title('Logistic Regression Confusion Matrix')
plt.show()


In [ ]:
# --- Step 10: Soft Voting Ensemble & confusion matrix ---
voter = VotingClassifier(
    estimators=[
        ('xgb', best_xgb),
        ('lgb', lgb_best),
        ('cat', cat_best)
    ],
    voting='soft',
    weights=[1,1,1]
)
print("\nTraining Voting Ensemble...")
voter.fit(X_train_scaled, y_train)
y_proba_v = voter.predict_proba(X_test_scaled)[:,1]
y_pred_v  = (y_proba_v > 0.5).astype(int)

print("\nEnsemble Metrics:")
print("AUC:              ", roc_auc_score(y_test, y_proba_v))
print("Balanced Accuracy:", balanced_accuracy_score(y_test, y_pred_v))
print(classification_report(y_test, y_pred_v, target_names=['Benign','Ovarian'], zero_division=0))

# Plot Ensemble confusion matrix
cm = confusion_matrix(y_test, y_pred_v)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['Benign','Ovarian'])
disp.plot(cmap=plt.cm.Blues)
plt.title('Voting Ensemble Confusion Matrix')
plt.show()


[CV] END colsample_bytree=0.6, gamma=0.5, learning_rate=0.2, max_depth=4, n_estimators=100, reg_alpha=0.1, reg_lambda=1.5, scale_pos_weight=1, subsample=1.0; total time=   0.2s
[CV] END colsample_bytree=0.6, gamma=0.5, learning_rate=0.2, max_depth=4, n_estimators=100, reg_alpha=0.1, reg_lambda=1.5, scale_pos_weight=1, subsample=1.0; total time=   0.1s
[CV] END border_count=128, depth=8, iterations=800, l2_leaf_reg=3, learning_rate=0.1, scale_pos_weight=1; total time=  35.7s
[CV] END border_count=64, depth=4, iterations=200, l2_leaf_reg=1, learning_rate=0.1, scale_pos_weight=1.105263157894737; total time=   1.5s
[CV] END border_count=64, depth=4, iterations=200, l2_leaf_reg=1, learning_rate=0.1, scale_pos_weight=1.105263157894737; total time=   1.7s
[CV] END border_count=128, depth=8, iterations=200, l2_leaf_reg=5, learning_rate=0.1, scale_pos_weight=1; total time=   8.8s
[CV] END border_count=128, depth=8, iterations=200, l2_leaf_reg=5, learning_rate=0.1, scale_pos_weight=1; total tim